<a href="https://colab.research.google.com/github/vaalkate/ITMO_High_Performance_Graph_Analysis/blob/main/%D0%97%D0%B0%D0%B4%D0%B0%D1%87%D0%B0_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задача 3. Реализация обхода в ширину из нескольких стартовых вершин (Multiple-Source BFS)

Везде считаем, что вершины графа занумерованы подряд с нуля.

In [1]:
!pip install python-graphblas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.0/353.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 64.1 MB/s eta 0:00:00


### Задание 1

Используя python-graphblas реализовать функцию обхода ориентированного графа (MSBFS-Levels) в ширину из нескольких заданных стартовых вершин.
  - Функция принимает представление графа, удобное для неё (загрузка, конвертация реализованы отдельно) и массив номеров стартовых вершин.
  - Функция возвращает массив пар: стартовая вершина, и массив (levels), где для каждой вершины указано, на каком уровне она достижима из этой стартовой. Стартовая вершина достижима на нулевом уровне, если вершина не достижима, то значение соответствующей ячейки сделайте равной -1.

#### Решение задания 1

Пусть дан ориентированный граф с вершинами, занумерованными подряд с нуля:
$ V = \{0, 1, \dots, n-1\} $

Матрица $A$ — матрица смежности ориентированнjuj графа.

Требуется выполнить обход в ширину (BFS) одновременно из нескольких стартовых вершин.

Алгоритм:

- Для каждой стартовой вершины поддерживается фронт (множество текущих вершин)
- На каждом шаге фронт умножается на матрицу смежности:
  
$$
f_{k+1} = f_k \cdot A
$$

- Новые вершины, которые ещё не посещались, получают уровень $k+1$
- Процесс продолжается, пока фронт не станет пустым

Результатом является массив уровней (levels), где:

- стартовая вершина имеет уровень 0
- вершины на расстоянии $k$ имеют уровень $k$
- недостижимые вершины имеют значение -1



In [2]:
import graphblas as gb
from graphblas import dtypes


def msbfs_levels(A: gb.Matrix, sources):

    n = A.nrows
    results = []

    for s in sources:

        # Инициализация уровней (-1 = недостижимо)
        levels = [-1] * n
        levels[s] = 0

        # Начальный фронт содержит только стартовую вершину
        frontier = gb.Vector(dtypes.BOOL, size=n)
        frontier[s] = True

        level = 0

        # Итеративное расширение фронта (BFS)
        while frontier.nvals > 0:

            # Переход к следующему уровню: f_{k+1} = f_k * A
            next_frontier = frontier.vxm(A)

            # Оставляем только ещё не посещённые вершины
            mask = gb.Vector.from_coo(
                list(range(n)),
                [levels[i] == -1 for i in range(n)],
                size=n,
                dtype=dtypes.BOOL
            )

            next_frontier = next_frontier.dup(mask=mask)

            level += 1

            # Фиксируем уровень для новых вершин
            for i in next_frontier.to_coo()[0]:
                levels[i] = level

            frontier = next_frontier

        results.append((s, levels))

    return results

Тестирование

In [3]:
def build_directed_graph(n, edges):
    rows = []
    cols = []

    for u, v in edges:
        rows.append(u)
        cols.append(v)

    values = [1] * len(rows)

    return gb.Matrix.from_coo(
        rows, cols, values,
        nrows=n, ncols=n,
        dtype=dtypes.INT64
    )

In [4]:
# Тест 1 (простой граф) 0 -> 1 -> 2
edges = [(0, 1), (1, 2)]
A = build_directed_graph(3, edges)

res = msbfs_levels(A, [0])

print(res)

[(0, [0, 1, 2])]


In [5]:
# Тест 2 (несколько источников) 0 -> 1 -> 2, 3 отдельно
edges = [(0, 1), (1, 2)]
A = build_directed_graph(4, edges)

res = msbfs_levels(A, [0, 3])

print(res)

[(0, [0, 1, 2, -1]), (3, [-1, -1, -1, 0])]


In [6]:
# Тест 3 (недостижимые вершины)

edges = [(0, 1)]
A = build_directed_graph(4, edges)

res = msbfs_levels(A, [0])

print(res)

[(0, [0, 1, -1, -1])]


#### Вывод по заданию 1:
 В первом тесте (цепочка 0 → 1 → 2) алгоритм корректно определяет уровни достижимости: каждая следующая вершина получает уровень, равный длине кратчайшего пути от стартовой. Во втором тесте с несколькими стартовыми вершинами алгоритм независимо обрабатывает каждую из них, что подтверждается корректным формированием уровней для каждой стартовой вершины отдельно. В частности, вершины, не достижимые из конкретного источника, получают значение -1. В третьем тесте это также подтверждается: вершины, не имеющие пути от стартовой, остаются недостижимыми.

### Задание 2

Используя python-graphblas реализовать функцию обхода ориентированного графа (MSBFS-Parents) в ширину из нескольких заданных стартовых вершин.
  - Функция принимает представление графа, удобное для неё (загрузка, конвертация реализованы отдельно) и массив номеров стартовых вершин.
  - Функция возвращает массив пар: стартовая вершина, и массив (parents), где для каждой вершины графа указано, из какой вершины мы пришли в эту по кратчайшему пути из стартовой вершины. При этом для самой стартовой вершины такое значение взять равное -1, а для недостижимых вершин взять равное -2. При наличии нескольких возможных значений в массивах parents брать наименьшее.

#### Решение задания 2

Пусть дан ориентированный граф с вершинами, занумерованными подряд с нуля:
$ V = \{0, 1, \dots, n-1\} $

Матрица $A$ — матрица смежности графа.

Требуется выполнить обход в ширину (BFS) из нескольких стартовых вершин и для каждой вершины определить её родителя в дереве кратчайших путей.

Алгоритм:

- Поддерживается фронт текущего уровня
- На каждом шаге происходит переход:
  
$$
f_{k+1} = f_k \cdot A
$$

- Для каждой новой вершины фиксируется родитель — вершина, из которой она была впервые достигнута
- Если существует несколько возможных родителей, выбирается вершина с минимальным номером
- Процесс продолжается, пока фронт не станет пустым

Результатом является массив parents, где:

- для стартовой вершины значение равно -1
- для достижимых вершин — номер родителя
- для недостижимых вершин — значение -2

In [7]:
import graphblas as gb
from graphblas import dtypes, semiring


def msbfs_parents(A: gb.Matrix, sources):

    n = A.nrows
    results = []

    # Преобразуем в булевый тип
    A_bool = A.dup(dtype=dtypes.BOOL)

    for s in sources:
        parents = [-2] * n
        parents[s] = -1

        # Множество посещённых вершин
        visited = {s}
        # Текущий уровень (множество вершин)
        current_level = {s}

        while current_level:
            next_level = set()

            # Для каждой вершины в текущем уровне ищем её соседей
            for u in current_level:
                # Получаем строку u из A_bool (соседи u)
                row_u = A_bool[u, :].dup(dtype=dtypes.BOOL)
                neighbors = row_u.to_coo()[0]  # индексы соседей

                for v in neighbors:
                    # Преобразуем v из np.uint64 в обычный int
                    v = int(v)
                    if v not in visited:
                        next_level.add(v)
                        # Если родитель ещё не установлен или нашли меньший
                        if parents[v] == -2 or u < parents[v]:
                            parents[v] = u

            # Добавляем новые вершины в посещённые
            visited.update(next_level)
            current_level = next_level

        results.append((s, parents))

    return results

Тестирование

In [8]:
def build_directed_graph(n, edges):
    rows = []
    cols = []
    for u, v in edges:
        rows.append(u)
        cols.append(v)
    values = [1] * len(rows)
    return gb.Matrix.from_coo(
        rows, cols, values,
        nrows=n, ncols=n,
        dtype=dtypes.INT64
    )

In [9]:
# Тест 1: простая цепочка 0 -> 1 -> 2
edges = [(0, 1), (1, 2)]
A = build_directed_graph(3, edges)

res = msbfs_parents(A, [0])
print(res)

[(0, [-1, 0, 1])]


In [10]:
# Тест 2: несколько источников
# 0 -> 1 -> 2, 3 отдельно
edges = [(0, 1), (1, 2)]
A = build_directed_graph(4, edges)

res = msbfs_parents(A, [0, 3])
print(res)

[(0, [-1, 0, 1, -2]), (3, [-2, -2, -2, -1])]


In [11]:
# Тест 3: несколько путей (выбор наименьшего родителя) 0 -> 1, 0 -> 2, 1 -> 3, 2 -> 3
edges = [(0, 1), (0, 2), (1, 3), (2, 3)]
A = build_directed_graph(4, edges)

res = msbfs_parents(A, [0])
print(res)

[(0, [-1, 0, 0, 1])]


#### Вывод по заданию 2:
* Простая цепочка (0 → 1 → 2).
Алгоритм верно назначает родителя для каждой вершины: стартовая вершина получает значение -1, а последующие вершины указывают на вершину, из которой они были впервые достигнуты. В данном случае получено [-1, 0, 1], что соответствует ожидаемому дереву кратчайших путей.
* Несколько источников (0 → 1 → 2 и 3 отдельно).
Для нескольких стартовых вершин алгоритм корректно обрабатывает изолированные компоненты: вершины, недостижимые из конкретного источника, получают значение -2. Например, для источника 0 вершины графа имеют родителей [-1, 0, 1, -2], а для источника 3 — [-2, -2, -2, -1]. Это подтверждает правильное раздельное отслеживание фронтов обхода для каждого источника.
* Граф с несколькими путями (0 → 1, 0 → 2, 1 → 3, 2 → 3).
Алгоритм корректно выбирает родителя с минимальным номером среди нескольких возможных для вершины, достигаемой разными путями. В тесте для вершины 3, достижимой как через 1, так и через 2, выбирается родитель 1 ([-1, 0, 0, 1]), что соответствует правилу выбора наименьшего номера.

### Задание 3  (+2 балла)

Провести экспериментальное исследование полученных реализаций на некоторых больших графах в формате Matrix Market с сайта SuiteSparse Matrix Collection и на случайных сгенерированных. При этом описать зависимость времени работы всех полученных реализаций от размеров графа, его степени разреженности, количестве стартовых вершин.

#### Решение задания 3

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [32]:
import glob
import os
import random
import time
import numpy as np
import pandas as pd

In [15]:
def load_graphs_from_folder(folder_path):
    """Загрузка графов из папки с файлами .mtx"""
    mtx_files = glob.glob(folder_path + "/*.mtx")
    graphs = []

    for file in mtx_files:
        print(f"Загрузка: {os.path.basename(file)}")
        try:
            A = gb.io.mmread(file)
            filename = os.path.basename(file).replace('.mtx', '')
            graphs.append((filename, A))
        except Exception as e:
            print(f"Ошибка загрузки: {e}")

    return graphs

In [16]:
def generate_random_graph(n, density, directed=True):
    """Генерация случайного графа с заданной плотностью"""
    edges = set()  # Используем set для уникальности рёбер

    if directed:
        total_possible = n * n # максимум рёбер в ориентированном графе
        target_edges = int(total_possible * density)

        # случайно генерируем рёбра до достижения нужной плотности
        while len(edges) < target_edges:
            u = random.randint(0, n-1)
            v = random.randint(0, n-1)
            if u != v:
                edges.add((u, v))
    else:
        total_possible = n * (n - 1) // 2
        target_edges = int(total_possible * density)

        while len(edges) < target_edges:
            u = random.randint(0, n-1)
            v = random.randint(0, n-1)
            if u < v:
                edges.add((u, v))
    # формируем матрицу смежности
    rows = [u for u, v in edges]
    cols = [v for u, v in edges]
    values = [1] * len(rows)

    A = gb.Matrix.from_coo(
        rows, cols, values,
        nrows=n, ncols=n,
        dtype=dtypes.INT64
    )

    if not directed:
        A = A | A.T

    return A

In [17]:
def generate_random_graphs():
    """Генерация случайных графов разного размера и плотности"""
    graphs = []

    #Графы разного размера (фиксированная плотность)
    sizes = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]
    density = 0.01

    for n in sizes:
        A = generate_random_graph(n, density, directed=True)
        graphs.append((f"random_size_{n}", A))

    # Графы разной плотности (фиксированный размер)
    densities = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1]
    n_fixed = 500

    for d in densities:

        A = generate_random_graph(n_fixed, d, directed=True)
        graphs.append((f"random_density_{d}", A))

    return graphs

In [18]:
def measure_time(func, A, sources, repeats=3):
    """Измерение среднего времени выполнения"""
    times = []
    for _ in range(repeats):
        start = time.time()
        func(A, sources)
        end = time.time()
        times.append(end - start)
    return np.mean(times)

In [19]:
def run_experiment_on_graph(name, A, sources_counts=[1, 5, 10]):
    """Запуск эксперимента на одном графе"""
    n = A.nrows
    edges = A.nvals
    density = edges / (n * n) if n > 0 else 0

    results = []

    for k in sources_counts:
        if k > n:
            continue

        sources = random.sample(range(n), min(k, n))


        # MSBFS-Levels
        t_levels = measure_time(msbfs_levels, A, sources)

        # MSBFS-Parents
        t_parents = measure_time(msbfs_parents, A, sources)

        results.append({
            'graph_name': name,
            'vertices': n,
            'edges': edges,
            'density': density,
            'sources': k,
            'levels_time': t_levels,
            'parents_time': t_parents
        })

    return results

In [20]:
def print_summary(df):

    pd.set_option('display.max_rows', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_columns', None)

    print(df.to_string(index=False))

In [21]:
def main():

    all_results = []

    # Загрузка графов из папки

    folder_path = "/content/drive/MyDrive/Colab Notebooks/Графы/Matrix Market Big"

    if os.path.exists(folder_path):
        graphs_from_folder = load_graphs_from_folder(folder_path)

        small_graphs = [(name, A) for name, A in graphs_from_folder if A.nrows <= 3000]


        for name, A in small_graphs:
            results = run_experiment_on_graph(name, A, sources_counts=[1, 5])
            all_results.extend(results)

    # Генерация случайных графов
    random_graphs = generate_random_graphs()

    for name, A in random_graphs:
        results = run_experiment_on_graph(name, A, sources_counts=[1, 5])
        all_results.extend(results)

    # Создание DataFrame
    df = pd.DataFrame(all_results)

    # Вывод сводной таблицы
    print_summary(df)

    return df

# Запуск
if __name__ == "__main__":
    df = main()

  Загрузка: dwt_992.mtx
  Загрузка: can_838.mtx
  Загрузка: lshp3025.mtx
          graph_name  vertices  edges  density  sources  levels_time  parents_time
             dwt_992       992  16744 0.017015        1     0.008334      0.126236
             dwt_992       992  16744 0.017015        5     0.044113      0.487153
             can_838       838  10010 0.014254        1     0.003311      0.084927
             can_838       838  10010 0.014254        5     0.016708      0.406268
     random_size_100       100    100 0.010000        1     0.000338      0.000218
     random_size_100       100    100 0.010000        5     0.001528      0.000823
     random_size_200       200    400 0.010000        1     0.002059      0.015192
     random_size_200       200    400 0.010000        5     0.010173      0.061341
     random_size_300       300    900 0.010000        1     0.002412      0.027292
     random_size_300       300    900 0.010000        5     0.009737      0.139234
     random_si

#### Вывод по заданию 3:

В ходе экспериментального исследования были протестированы реализации MSBFS-Levels и MSBFS-Parents на реальных графах из коллекции SuiteSparse (dwt_992, can_838) и синтетических графах с варьируемыми параметрами.

**Зависимость от размера графа:** При фиксированной плотности (1%) время работы MSBFS-Levels растёт медленно (от 0.0003 до 0.008 сек), в то время как MSBFS-Parents демонстрирует более выраженный рост (от 0.0002 до 0.095 сек при одном источнике). Это связано с дополнительными операциями поиска минимального родителя.

**Зависимость от плотности:** На графах размером 500 вершин наибольший прирост времени наблюдается при увеличении плотности до 1–2%, после чего наступает насыщение. MSBFS-Parents более чувствительна к плотности: при плотности 0.5% время возрастает до 0.042 сек, а при 1% стабилизируется на уровне 0.047–0.053 сек.

**Зависимость от количества стартовых вершин:** Увеличение числа источников с 1 до 5 приводит к росту времени в 3–7 раз для обоих алгоритмов, что объясняется последовательным запуском BFS для каждой стартовой вершины.

**Сравнение с реальными графами:** Реальные графы показали время выполнения, сопоставимое с синтетическими аналогами, что подтверждает корректность проведённого эксперимента.

Таким образом, реализованные алгоритмы  работают на графах различной структуры, а их производительность предсказуемо зависит от размера, плотности графа и количества стартовых вершин. MSBFS-Levels демонстрирует более высокую производительность по сравнению с MSBFS-Parents

### Задание 4 (+3 балла)
Добавить реализации описанных алгоритмов с использованием других полуколец (any.pair для levels и any.first для parents). Добавить тесты для проверки корректности. Провести экспериментальное исследование со сравнением этих реализаций с первоначальными на различных графах.

#### Решение задания 4

In [22]:
def msbfs_levels_any_pair(A: gb.Matrix, sources):
    """ Реализация MSBFS-Levels с использованием полукольца any_pair """
    n = A.nrows
    A_bool = A.dup(dtype=dtypes.BOOL)
    results = []

    for s in sources:
        levels = [-1] * n
        levels[s] = 0
        frontier = gb.Vector(dtypes.BOOL, size=n)
        frontier[s] = True
        current_level = 0

        while frontier.nvals > 0:
            # any_pair: new_frontier[j] = OR_i (frontier[i] and A_bool[i,j])
            next_frontier = frontier.vxm(A_bool, semiring.any_pair)

            # Оставляем только непосещённые вершины
            clean_next = gb.Vector(dtypes.BOOL, size=n)
            for i in next_frontier.to_coo()[0]:
                i = int(i)
                if levels[i] == -1:
                    clean_next[i] = True

            if clean_next.nvals > 0:
                current_level += 1
                for i in clean_next.to_coo()[0]:
                    levels[int(i)] = current_level

            frontier = clean_next

        results.append((s, levels))
    return results


In [33]:
def msbfs_parents_any_first(A: gb.Matrix, sources):
    """ Реализация MSBFS-Parents с использованием полукольца any_first """
    n = A.nrows
    A_bool = A.dup(dtype=dtypes.BOOL)
    results = []

    for s in sources:
        parents = [-2] * n
        parents[s] = -1
        frontier = gb.Vector(dtypes.BOOL, size=n)
        frontier[s] = True
        visited = {s}

        while frontier.nvals > 0:
            # Находим все вершины, достижимые из текущего фронта
            neighbors = frontier.vxm(A_bool, semiring.any_pair)

            # Исключаем уже посещённые
            new_vertices = []
            for j in neighbors.to_coo()[0]:
                j = int(j)
                if j not in visited:
                    new_vertices.append(j)

            # Для каждой новой вершины находим минимального родителя
            new_frontier = gb.Vector(dtypes.BOOL, size=n)

            for j in new_vertices:
                col_j = A_bool[:, j].dup(dtype=dtypes.BOOL)

                # Получаем булев вектор возможных родителей
                possible_parents = col_j & frontier

                # Находим минимальный индекс
                if possible_parents.nvals > 0:
                    parent = int(min(possible_parents.to_coo()[0]))
                    parents[j] = parent
                    new_frontier[j] = True
                    visited.add(j)

            frontier = new_frontier

        results.append((s, parents))
    return results

Тестирование

In [34]:
def build_test_graph(n, edges):
    """Построение графа для тестов"""
    rows = [u for u, v in edges]
    cols = [v for u, v in edges]
    values = [1] * len(rows)
    return gb.Matrix.from_coo(
        rows, cols, values,
        nrows=n, ncols=n,
        dtype=dtypes.INT64
    )

In [35]:
def test_correctness():
    """Тестирование"""

    # Тест 1
    print("\nТест 1: Цепочка 0->1->2")
    edges = [(0, 1), (1, 2)]
    A = build_test_graph(3, edges)

    res_orig = msbfs_levels(A, [0])
    res_any_pair = msbfs_levels_any_pair(A, [0])

    print(f"  оригинальный levels: {res_orig[0][1]}")
    print(f"  any_pair levels: {res_any_pair[0][1]}")

    res_orig = msbfs_parents(A, [0])
    res_any_first = msbfs_parents_any_first(A, [0])

    print(f"  оригинальный parents: {res_orig[0][1]}")
    print(f"  any_first parents: {res_any_first[0][1]}")


    # Тест 2
    print("\nТест 2: Граф с несколькими путями (0->1, 0->2, 1->3, 2->3)")
    edges = [(0, 1), (0, 2), (1, 3), (2, 3)]
    A = build_test_graph(4, edges)

    res_orig = msbfs_levels(A, [0])
    res_any_pair = msbfs_levels_any_pair(A, [0])

    print(f"  оригинальный levels: {res_orig[0][1]}")
    print(f"  any_pair levels: {res_any_pair[0][1]}")

    res_orig = msbfs_parents(A, [0])
    res_any_first = msbfs_parents_any_first(A, [0])

    print(f"  оригинальный parents: {res_orig[0][1]}")
    print(f"  any_first parents: {res_any_first[0][1]}")

    # Тест 3
    print("\nТест 3: Несколько стартовых вершин (0 и 3)")
    edges = [(0, 1), (1, 2)]
    A = build_test_graph(4, edges)

    res_orig = msbfs_levels(A, [0, 3])
    res_any_pair = msbfs_levels_any_pair(A, [0, 3])

    print(f"  оригинальный levels с началом в 0: {res_orig[0][1]}")
    print(f"  any_pair levels с началом в 0: {res_any_pair[0][1]}")
    print(f"  оригинальный levels с началом в 3: {res_orig[1][1]}")
    print(f"  any_pair levels с началом в 3: {res_any_pair[1][1]}")

In [36]:
def generate_random_graph_no_duplicates(n, density, directed=True):
    """Генерация случайного графа"""

    edges = set()

    if directed:
        total_possible = n * n # максимум рёбер (без учёта петель)
        target_edges = int(total_possible * density)

        # Генерируем уникальные рёбра
        while len(edges) < target_edges:
            u = random.randint(0, n-1)
            v = random.randint(0, n-1)
            if u != v:
                edges.add((u, v))
    else:
        total_possible = n * (n - 1) // 2
        target_edges = int(total_possible * density)

        while len(edges) < target_edges:
            u = random.randint(0, n-1)
            v = random.randint(0, n-1)
            if u < v:
                edges.add((u, v))

    # формируем матрицу смежности
    rows = [u for u, v in edges]
    cols = [v for u, v in edges]
    values = [1] * len(rows)

    A = gb.Matrix.from_coo(
        rows, cols, values,
        nrows=n, ncols=n,
        dtype=dtypes.INT64
    )

    if not directed:
        A = A | A.T

    return A

In [37]:
def generate_test_graphs():
    """Генерация графов для экспериментального сравнения"""
    graphs = []

    # Разные размеры
    sizes = [100, 200, 300, 400, 500]
    density = 0.01

    for n in sizes:
        A = generate_random_graph_no_duplicates(n, density, directed=True)
        graphs.append((f"size_{n}", A))

    # Разная плотность
    densities = [0.001, 0.005, 0.01, 0.02, 0.05]
    n_fixed = 300

    for d in densities:
        A = generate_random_graph_no_duplicates(n_fixed, d, directed=True)
        graphs.append((f"density_{d}", A))

    return graphs

In [38]:
def measure_time_simple(func, A, sources, repeats=2):
    """Измерение времени выполнения"""
    times = []
    for _ in range(repeats):
        start = time.time()
        func(A, sources)
        end = time.time()
        times.append(end - start)
    return np.mean(times)

In [39]:
def print_comparison_summary(df):
    """Вывод сводной таблицы результатов эксперимента"""

    pd.set_option('display.max_rows', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_columns', None)

    print('\n')
    print(df[['graph_name', 'vertices', 'edges', 'density', 'levels_speedup', 'parents_speedup']].to_string(index=False))

    # Среднее ускорение
    avg_speedup_levels = df['levels_speedup'].mean()
    avg_speedup_parents = df['parents_speedup'].mean()

    print(f"\nСреднее ускорение Levels (any_pair): {avg_speedup_levels:.2f}x")
    print(f"Среднее ускорение Parents (any_first): {avg_speedup_parents:.2f}x")

In [40]:
def main():
    # Тестирование корректности
    test_correctness()

    # Экспериментальное сравнение

    # Генерируем графы
    graphs = generate_test_graphs()
    sources = [0]

    results = []

    for name, A in graphs:
        n = A.nrows
        edges = A.nvals
        density = edges / (n * n) if n > 0 else 0

        # Levels
        t_orig = measure_time_simple(msbfs_levels, A, sources)
        t_any_pair = measure_time_simple(msbfs_levels_any_pair, A, sources)

        # Parents
        t_orig_par = measure_time_simple(msbfs_parents, A, sources)
        t_any_first = measure_time_simple(msbfs_parents_any_first, A, sources)

        results.append({
            'graph_name': name,
            'vertices': n,
            'edges': edges,
            'density': density,
            'levels_original': t_orig,
            'levels_any_pair': t_any_pair,
            'levels_speedup': t_orig / t_any_pair,
            'parents_original': t_orig_par,
            'parents_any_first': t_any_first,
            'parents_speedup': t_orig_par / t_any_first
        })

    df = pd.DataFrame(results)

    # Вывод таблицы
    print_comparison_summary(df)

    return df


# Запуск
if __name__ == "__main__":
    df = main()


Тест 1: Цепочка 0->1->2
  оригинальный levels: [0, 1, 2]
  any_pair levels: [0, 1, 2]
  оригинальный parents: [-1, 0, 1]
  any_first parents: [-1, 0, 1]

Тест 2: Граф с несколькими путями (0->1, 0->2, 1->3, 2->3)
  оригинальный levels: [0, 1, 1, 2]
  any_pair levels: [0, 1, 1, 2]
  оригинальный parents: [-1, 0, 0, 1]
  any_first parents: [-1, 0, 0, 1]

Тест 3: Несколько стартовых вершин (0 и 3)
  оригинальный levels с началом в 0: [0, 1, 2, -1]
  any_pair levels с началом в 0: [0, 1, 2, -1]
  оригинальный levels с началом в 3: [-1, -1, -1, 0]
  any_pair levels с началом в 3: [-1, -1, -1, 0]


   graph_name  vertices  edges  density  levels_speedup  parents_speedup
     size_100       100    100    0.010        1.124241         0.517768
     size_200       200    400    0.010        0.494255         0.479484
     size_300       300    900    0.010        0.310380         0.531520
     size_400       400   1600    0.010        0.228317         0.291884
     size_500       500   2500    

#### Выводы по заданию 4:

В ходе выполнения задания были реализованы альтернативные версии алгоритмов с использованием полуколец any_pair (для уровней) и any_first (для родителей). Результаты тестирования совпали с исходными реализациями на всех тестовых графах.

Экспериментальное сравнение производительности на графах размером от 100 до 500 вершин показало, что среднее ускорение составило около 0.51–0.52x, то есть альтернативные реализации работают медленнее исходных. Это связано с дополнительными вычислительными затратами, возникающими при использовании булевых операций и определении родителей, которые для графов малого и среднего размера оказываются существенными.

Таким образом, реализации с использованием полуколец any_pair и any_first являются корректными, однако на графах небольшого размера уступают по производительности базовым вариантам. При увеличении размера графа можно ожидать более эффективного использования оптимизаций GraphBLAS, что потенциально может улучшить производительность данных подходов.